## Customer Analytics - Creación de bases de datos

### 1. Importación de librerías y configuración para crear las bases de datos 

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake_es = Faker('es_ES')
np.random.seed(42)
random.seed(42)
os.makedirs('data/raw', exist_ok=True)

In [4]:
from faker import Faker
import random

fake = Faker('es_ES')

In [2]:
# ── PARÁMETROS ─────────────────────────────────────────────
N_CLIENTES    = 1_000
N_TRANSACC    = 8_000
FECHA_INICIO  = datetime(2023, 1, 1)
FECHA_FIN     = datetime(2025, 12, 31)

CIUDADES = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao',
            'Zaragoza', 'Málaga', 'Alicante']

CANALES_ADQUISICION = ['organico', 'paid', 'referral', 'tienda']

CATEGORIAS = ['Montaña', 'Running', 'Ciclismo', 'Padel']

TIENDAS = [
    ('T01', 'Cumbres Madrid',     'Madrid'),
    ('T02', 'Cumbres Barcelona',  'Barcelona'),
    ('T03', 'Cumbres Valencia',   'Valencia'),
    ('T04', 'Cumbres Sevilla',    'Sevilla'),
    ('T05', 'Cumbres Web',        'Online'),
]


# ── HELPERS ────────────────────────────────────────────────
def fecha_random():
    delta = (FECHA_FIN - FECHA_INICIO).days
    return FECHA_INICIO + timedelta(days=random.randint(0, delta))




## 2. Creación de dataframes

In [8]:
# ── 1. TIENDAS ─────────────────────────────────────────────
tiendas = pd.DataFrame(TIENDAS, columns=['store_id', 'nombre', 'ciudad'])

In [8]:
tiendas.head()

,store_id,nombre,ciudad
0,T01,Cumbres Madrid,Madrid
1,T02,Cumbres Barcelona,Barcelona
2,T03,Cumbres Valencia,Valencia
3,T04,Cumbres Sevilla,Sevilla
4,T05,Cumbres Web,Online


In [5]:
# ── 2. PRODUCTOS ───────────────────────────────────────────
productos = pd.DataFrame([
    {
        'product_id':  f"P{str(i).zfill(3)}",
        'nombre':      f"{fake.word().capitalize()} {cat}",
        'categoria':   cat,
        'precio_base': round(random.choice([
            random.uniform(15, 60),    # accesorios
            random.uniform(60, 180),   # ropa técnica
            random.uniform(180, 500),  # equipamiento
        ]), 2),
    }
    for i, cat in enumerate(
        [c for c in CATEGORIAS for _ in range(12)], start=1
    )
])


In [6]:
# ── 3. CLIENTES ────────────────────────────────────────────
clientes = pd.DataFrame([
    {
        'customer_id':       f"C{str(i).zfill(4)}",
        'nombre':            fake.first_name() + ' ' + fake.last_name(),
        'email':             fake.email(),
        'ciudad':            random.choice(CIUDADES),
        'canal_adquisicion': random.choice(CANALES_ADQUISICION),
        'fecha_alta':        fecha_random().strftime('%Y-%m-%d'),
    }
    for i in range(1, N_CLIENTES + 1)
])

# Suciedad mínima y documentada
clientes.loc[clientes.sample(frac=0.04).index, 'email']  = None  # emails sin registrar
clientes.loc[clientes.sample(frac=0.05).index, 'ciudad'] = None  # ciudad no informada

In [9]:
# ── 4. TRANSACCIONES ───────────────────────────────────────
ids_cliente  = clientes['customer_id'].tolist()
ids_producto = productos['product_id'].tolist()
ids_tienda   = tiendas['store_id'].tolist()

# Distribución realista: algunos clientes compran más que otros
pesos = np.random.exponential(1, N_CLIENTES)
pesos /= pesos.sum()

transacciones = pd.DataFrame([
    {
        'transaction_id': f"TX{str(i).zfill(5)}",
        'customer_id':    np.random.choice(ids_cliente, p=pesos),
        'product_id':     random.choice(ids_producto),
        'store_id':       random.choice(ids_tienda),
        'cantidad':       random.choices([1, 2, 3], weights=[0.70, 0.22, 0.08])[0],
        'importe':        round(random.uniform(20, 400), 2),
        'fecha':          fecha_random().strftime('%Y-%m-%d'),
    }
    for i in range(1, N_TRANSACC + 1)
])

# Suciedad mínima y documentada
transacciones.loc[transacciones.sample(frac=0.02).index, 'importe'] *= 8   # outliers POS
transacciones.loc[transacciones.sample(frac=0.01).index, 'importe'] *= -1  # devoluciones mal registradas
transacciones.loc[transacciones.sample(frac=0.03).index, 'store_id'] = None


In [10]:
print("── Cumbres Outdoor · Datos generados ──────────────────")
print(f"  Clientes       : {len(clientes):>5} filas")
print(f"  Transacciones  : {len(transacciones):>5} filas")
print(f"  Productos      : {len(productos):>5} filas")
print(f"  Tiendas        : {len(tiendas):>5} filas")
print()
print("── Suciedad intencional ────────────────────────────────")
for nombre, df in [('clientes', clientes), ('transacciones', transacciones)]:
    nulos = df.isnull().sum().sum()
    print(f"  {nombre:15s}: {nulos} nulos ({nulos/df.size*100:.1f}%)")
print()

── Cumbres Outdoor · Datos generados ──────────────────
  Clientes       :  1000 filas
  Transacciones  :  8000 filas
  Productos      :    48 filas
  Tiendas        :     5 filas

── Suciedad intencional ────────────────────────────────
  clientes       : 90 nulos (1.5%)
  transacciones  : 240 nulos (0.4%)



### 2.1 Importar archivos .csv

In [11]:
import os

# ── RUTAS ──────────────────────────────────────────────────
RAW_PATH   = r"C:\Users\marti\Desktop\Customer_analytics\data\raw"
CLEAN_PATH = r"C:\Users\marti\Desktop\Customer_analytics\data\clean"

# Crear carpetas si no existen
os.makedirs(RAW_PATH,   exist_ok=True)
os.makedirs(CLEAN_PATH, exist_ok=True)

In [13]:
tiendas.to_csv(r'C:\Users\marti\Desktop\Customer_analytics\data\raw\tiendas.csv', index=False)
productos.to_csv(r'C:\Users\marti\Desktop\Customer_analytics\data\raw\productos.csv', index=False)
clientes.to_csv(r'C:\Users\marti\Desktop\Customer_analytics\data\raw\clientes.csv', index=False)
transacciones.to_csv(r'C:\Users\marti\Desktop\Customer_analytics\data\raw\transacciones.csv', index=False)